In [1]:
# ============================================================
# TRUSTSYN TRUST LAYER v3
# Conformal Prediction Intervals
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

BASE = Path("/Users/konuri/stacking")

INPUT = (
    BASE /
    "TrustSyn_TRUST_LAYER" /
    "TrustSyn_novelty_scores.csv"
)

OUTPUT = (
    BASE /
    "TrustSyn_TRUST_LAYER" /
    "TrustSyn_conformal_predictions.csv"
)


# ------------------------------------------------------------
# LOAD TRUST TABLE
# ------------------------------------------------------------

df = pd.read_csv(INPUT)

print("Loaded:")
print(df.shape)



# ------------------------------------------------------------
# CALIBRATION ERROR
# ------------------------------------------------------------

print("\nSearching calibration files...")


calibration_files = [

    BASE /
    "STACKING_TABLES" /
    "RANDOM_STACKING_TABLE.csv",

    BASE /
    "STACKING_TABLES" /
    "COLD_COMBINATION_STACKING_TABLE.csv",

    BASE /
    "STACKING_TABLES" /
    "COLD_CELL_STACKING_TABLE.csv"
]


errors = []


for file in calibration_files:

    if file.exists():

        temp = pd.read_csv(file)

        print(
            "Using:",
            file.name,
            temp.shape
        )


        if (
            "y_true" in temp.columns
            and
            "dmpnn_prediction" in temp.columns
            and
            "catboost_prediction" in temp.columns
        ):

            ensemble = (
                temp[
                    [
                        "catboost_prediction",
                        "dmpnn_prediction"
                    ]
                ]
                .mean(axis=1)
            )


            err = np.abs(
                temp["y_true"]
                -
                ensemble
            )


            errors.extend(
                err.tolist()
            )


if len(errors) == 0:

    raise Exception(
        "No calibration errors found"
    )


errors = np.array(errors)



# ------------------------------------------------------------
# 90% CONFORMAL ERROR
# ------------------------------------------------------------

q90 = np.quantile(
    errors,
    0.90
)


print(
    "\n90% calibration error:",
    q90
)



# ------------------------------------------------------------
# APPLY INTERVALS
# ------------------------------------------------------------

df["conformal_lower_90"] = (
    df["ensemble_mean"]
    -
    q90
)


df["conformal_upper_90"] = (
    df["ensemble_mean"]
    +
    q90
)


df["confidence_level"] = "90%"



# interval width

df["prediction_interval_width"] = (
    df["conformal_upper_90"]
    -
    df["conformal_lower_90"]
)



# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

df.to_csv(
    OUTPUT,
    index=False
)


print("\nFINAL TRUST LAYER v3")
print(df.shape)


print(
    df[
        [
            "ensemble_mean",
            "conformal_lower_90",
            "conformal_upper_90",
            "prediction_interval_width"
        ]
    ].head()
)


print("\nSAVED:")
print(OUTPUT)

Loaded:
(88753, 82)

Searching calibration files...
Using: RANDOM_STACKING_TABLE.csv (29408, 74)
Using: COLD_COMBINATION_STACKING_TABLE.csv (29558, 74)
Using: COLD_CELL_STACKING_TABLE.csv (29787, 74)

90% calibration error: 9.602859414679571

FINAL TRUST LAYER v3
(88753, 86)
   ensemble_mean  conformal_lower_90  conformal_upper_90  \
0      -0.900527          -10.503387            8.702332   
1       0.044853           -9.558006            9.647713   
2      -1.269751          -10.872610            8.333108   
3      -2.126269          -11.729129            7.476590   
4      -0.796816          -10.399675            8.806044   

   prediction_interval_width  
0                  19.205719  
1                  19.205719  
2                  19.205719  
3                  19.205719  
4                  19.205719  

SAVED:
/Users/konuri/stacking/TrustSyn_TRUST_LAYER/TrustSyn_conformal_predictions.csv
